In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs('Time_inference', exist_ok=True)

df = pd.read_csv('Data.csv')
df = df.rename(columns={'F1 score': 'F1_Score', 'Inference Time (ms) median': 'Inference_Time_ms', 'Inference Time (ms) std': 'Inference_Time_Std_ms', 'Features': 'n_Features'})

datasets = df['Dataset'].unique()
methods = df['Method'].unique()
colors = ['#2F4F4F', '#007377', '#8B0000', '#B8860B', '#4B6C84', '#A0522D', '#556B2F']

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(datasets))
width = 0.12
offsets = np.linspace(-width * (len(methods) - 1) / 2, width * (len(methods) - 1) / 2, len(methods))

for i, method in enumerate(methods):
    subset = df[df['Method'] == method].set_index('Dataset').reindex(datasets)
    means = subset['Inference_Time_ms'].values
    stds = subset['Inference_Time_Std_ms'].values
    ax.bar(x + offsets[i], means, width, label=method, yerr=stds, capsize=4,
           color=colors[i % len(colors)], edgecolor='black', linewidth=0.5)

ax.set_xlabel('Dataset', fontsize=13)
ax.set_ylabel('Inference Time (ms)', fontsize=13)
ax.set_title('Inference Time per Dataset and Method (Mean \u00b1 Std)', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=45, ha='right', fontsize=11)
ax.legend(title='Method', fontsize=10, title_fontsize=11)
ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('Time_inference/bar_inference_time.svg', format='svg', bbox_inches='tight')
plt.close()
print('Saved bar_inference_time.svg')

Saved bar_inference_time.svg


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import pi

df = pd.read_csv('Data.csv')
df = df.rename(columns={'F1 score': 'F1_Score', 'Inference Time (ms) median': 'Inference_Time_ms', 'Inference Time (ms) std': 'Inference_Time_Std_ms', 'Features': 'n_Features'})

good_metrics = ['Accuracy', 'F1_Score']
bad_metrics = ['Leaves', 'Nodes', 'Depth', 'n_Features', 'Inference_Time_ms']
all_metrics = good_metrics + bad_metrics
num_vars = len(all_metrics)

datasets = df['Dataset'].unique()
colors = plt.cm.tab10.colors

for dataset in datasets:
    subset = df[df['Dataset'] == dataset].copy().reset_index(drop=True)
    methods = subset['Method'].values
    normalized = pd.DataFrame()
    normalized['Method'] = methods

    for metric in good_metrics:
        normalized[metric] = subset[metric].clip(0, 1)

    max_nodes = subset['Nodes'].max()
    normalized['Nodes'] = 1 - ((subset['Nodes'] - 1) / (max_nodes + 49 + 1e-10))

    max_leaves = subset['Leaves'].max()
    normalized['Leaves'] = 1 - ((subset['Leaves'] - 1) / (max_leaves + 24 + 1e-10))

    max_depth = subset['Depth'].max()
    normalized['Depth'] = 1 - ((subset['Depth'] - 1) / (max_depth + 9 + 1e-10))

    max_feat = subset['n_Features'].max()
    worst_feat = 1.25 * max_feat
    normalized['n_Features'] = 1 - ((subset['n_Features'] - 1) / (worst_feat - 1 + 1e-10))

    max_time = subset['Inference_Time_ms'].max()
    penalty_time = 2.5 * max_time
    worst_time = max_time + penalty_time
    normalized['Inference_Time_ms'] = 1 - (subset['Inference_Time_ms'] / (worst_time + 1e-10))

    angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    plt.title(f'Radar Plot \u2013 {dataset}', size=14, y=1.08)

    for i, method in enumerate(methods):
        values = normalized[normalized['Method'] == method][all_metrics].values.flatten().tolist()
        values += values[:1]
        color = colors[i % len(colors)]
        ax.plot(angles, values, color=color, linewidth=2, label=method)
        ax.fill(angles, values, color=color, alpha=0.25)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(all_metrics, fontsize=11)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['20%', '40%', '60%', '80%', '100%'], fontsize=10)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    fname = f'radar_{dataset.lower().replace(" ", "")}.svg'
    plt.savefig(f'Time_inference/{fname}', format='svg', bbox_inches='tight')
    plt.close()
    print(f'Saved {fname}')

Saved radar_banknoteauth.svg


Saved radar_breastcancer.svg


Saved radar_heartdisease.svg


Saved radar_ionosphere.svg


Saved radar_kc2.svg


Saved radar_qsarbiodeg.svg


Saved radar_spambase.svg


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import pi, ceil
from matplotlib.patches import Patch

df = pd.read_csv('Data.csv')
df = df.rename(columns={'F1 score': 'F1_Score', 'Inference Time (ms) median': 'Inference_Time_ms', 'Inference Time (ms) std': 'Inference_Time_Std_ms', 'Features': 'n_Features'})

good_metrics = ['Accuracy', 'F1_Score']
bad_metrics = ['Leaves', 'Nodes', 'Depth', 'n_Features', 'Inference_Time_ms']
all_metrics = good_metrics + bad_metrics
num_vars = len(all_metrics)

metric_labels = {
    'Accuracy': 'Acc', 'F1_Score': 'F1', 'Leaves': 'Leaves',
    'Nodes': 'Nodes', 'Depth': 'Depth', 'n_Features': 'Features',
    'Inference_Time_ms': 'Time(ms)'
}

angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]
angle_degrees = [a * 180 / pi for a in angles[:-1]]

formal_colors = ['#1f3c88', '#4a4a4a', '#2a7f62', '#a6233f', '#d55e00', '#54278f', '#007f7f']

datasets = df['Dataset'].unique()
n_datasets = len(datasets)
n_cols = 2
n_rows = ceil(n_datasets / n_cols)
legend_required = n_datasets % 2 != 0

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 7 * n_rows), subplot_kw=dict(polar=True))
axes = axes.flatten()

for idx, dataset in enumerate(datasets):
    subset = df[df['Dataset'] == dataset].copy().reset_index(drop=True)
    methods = subset['Method'].values
    normalized = pd.DataFrame()
    normalized['Method'] = methods

    for metric in good_metrics:
        normalized[metric] = subset[metric].clip(0, 1)

    max_nodes = subset['Nodes'].max()
    normalized['Nodes'] = 1 - ((subset['Nodes'] - 1) / (max_nodes + 49 + 1e-10))
    max_leaves = subset['Leaves'].max()
    normalized['Leaves'] = 1 - ((subset['Leaves'] - 1) / (max_leaves + 24 + 1e-10))
    max_depth = subset['Depth'].max()
    normalized['Depth'] = 1 - ((subset['Depth'] - 1) / (max_depth + 9 + 1e-10))
    max_feat = subset['n_Features'].max()
    worst_feat = 1.25 * max_feat
    normalized['n_Features'] = 1 - ((subset['n_Features'] - 1) / (worst_feat - 1 + 1e-10))
    max_time = subset['Inference_Time_ms'].max()
    penalty_time = 2.5 * max_time
    worst_time = max_time + penalty_time
    normalized['Inference_Time_ms'] = 1 - (subset['Inference_Time_ms'] / (worst_time + 1e-10))

    ax = axes[idx]
    ax.set_title(f'{dataset}', size=14, y=1.08)

    for i, method in enumerate(methods):
        values = normalized[normalized['Method'] == method][all_metrics].values.flatten().tolist()
        values += values[:1]
        color = formal_colors[i % len(formal_colors)]
        ax.plot(angles, values, color=color, linewidth=2, label=method)
        ax.fill(angles, values, color=color, alpha=0.15)

    ax.set_thetagrids(angle_degrees, labels=[metric_labels[m] for m in all_metrics], fontsize=12)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=10)
    ax.set_ylim(0, 1)

if legend_required:
    fig.delaxes(axes[-1])
    handles = [Patch(color=formal_colors[i % len(formal_colors)], label=method)
               for i, method in enumerate(df['Method'].unique())]
    legend_ax = fig.add_subplot(n_rows, n_cols, n_rows * 2)
    legend_ax.axis('off')
    legend_ax.legend(handles=handles, loc='center', fontsize=14, title='Methods',
                     title_fontsize=16, ncol=1, handlelength=2.5, labelspacing=1.2, borderpad=1.2)

plt.subplots_adjust(hspace=0.3)
plt.savefig('Time_inference/radar_grid.svg', format='svg', bbox_inches='tight')
plt.close()
print('Saved radar_grid.svg')

Saved radar_grid.svg
